In [ ]:
import numpy as np
import torch as th
from PIL import Image
import os


adjust_constrast = ["1.0", "1.5", "2.0", "2.5", "3.0", "3.5", "4.0"]
# adjust_constrast = [(x, y) for x in adjust_constrast for y in adjust_constrast]

fi = 57

def proc(c):
    row_res_grid = []
    row_ren_grid = []
    for i in adjust_constrast:
        row_res = []
        row_ren = []
        for j in adjust_constrast:
            down, up = i, j
            model = f"log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_SD256_adjcon{down}-{up}_rot2_{c}C"
            path = f"/data/mint/sampling/TPAMI_MajorRevision/FixPlastic/{model}/ema_300000/valid/render_face/reverse_sampling/src=66943.jpg/dst=68146.jpg/Lerp_1000/n_frames=60/"
            ren_img_path = f"{path}/dst_ren_frame{fi}.png"
            res_img_path = f"{path}/res_frame{fi}.png"
            if os.path.exists(res_img_path):
                res_img = Image.open(res_img_path)
                res_img = np.array(res_img)
            else:
                res_img = np.zeros((256, 256, 3), dtype=np.uint8) * 255
            
            if os.path.exists(ren_img_path):
                ren_img = Image.open(ren_img_path)
                ren_img = np.array(ren_img)
            else:
                ren_img = np.zeros((256, 256, 3), dtype=np.uint8) * 255
                
            row_res.append(res_img)
            row_ren.append(ren_img)
        row_res_grid.append(np.concatenate(row_res, axis=1))
        row_ren_grid.append(np.concatenate(row_ren, axis=1))
        
    row_res_grid = np.concatenate(row_res_grid, axis=0)
    row_res_grid = Image.fromarray(row_res_grid)
    row_res_grid.save(f"ACgrid_{c}C_fi{fi}_res.png")
    row_ren_grid = np.concatenate(row_ren_grid, axis=0)
    row_ren_grid = Image.fromarray(row_ren_grid)
    row_ren_grid.save(f"ACgrid_{c}C_fi{fi}_ren.png")


cc = ["0.0", "0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0"]
for c in cc:
    proc(c)

In [ ]:
import glob
import torchvision

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

nf = 60
def proc_vid(c):
    row_res_grid = []
    row_ren_grid = []
    for i in adjust_constrast:
        row_res = []
        row_ren = []
        for j in adjust_constrast:
            down, up = i, j
            model = f"log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_SD256_adjcon{down}-{up}_rot2_{c}C"
            path = f"/data/mint/sampling/TPAMI_MajorRevision/FixPlastic/{model}/ema_300000/valid/render_face/reverse_sampling/src=66943.jpg/dst=68146.jpg/Lerp_1000/n_frames={nf}/"
            ren_img_path = sort_by_frame(glob.glob(f"{path}/dst_ren_frame*.png"))
            res_img_path = sort_by_frame(glob.glob(f"{path}/res_frame*.png"))
            if len(res_img_path) > 0:
                ren_img = [np.array(Image.open(f)) for f in res_img_path][1:]
                res_img = np.stack(ren_img, axis=0)
            else:
                res_img = np.zeros((nf-1, 256, 256, 3), dtype=np.uint8) * 255

            if len(ren_img_path) > 0:
                ren_img = [np.array(Image.open(f)) for f in ren_img_path][1:]
                ren_img = np.stack(ren_img, axis=0)
            else:
                ren_img = np.zeros((nf-1, 256, 256, 3), dtype=np.uint8) * 255

            row_res.append(res_img)
            row_ren.append(ren_img)
        row_res_grid.append(np.concatenate(row_res, axis=2))
        row_ren_grid.append(np.concatenate(row_ren, axis=2))
        
    row_res_grid = np.concatenate(row_res_grid, axis=1)
    torchvision.io.write_video(f"ACgrid_{c}C_res.mp4", th.from_numpy(np.array(row_res_grid)), fps=24, video_codec='libx264', options={"crf": "17", "preset": "veryslow"})
    # row_res_grid.save(f"ACgrid_{c}C_fi{fi}_res.png")
    row_ren_grid = np.concatenate(row_ren_grid, axis=1)
    torchvision.io.write_video(f"ACgrid_{c}C_ren.mp4", th.from_numpy(np.array(row_ren_grid)), fps=24, video_codec='libx264', options={"crf": "17", "preset": "veryslow"})
    # row_ren_grid.save(f"ACgrid_{c}C_fi{fi}_ren.png")
    
for c in cc:
    proc_vid(c)


KeyboardInterrupt: 